____
### 0. Preamble
This notebook serves as a continuation of my other notebook file `autocomplete.ipynb` which trained a model within a simple linear model. 
Within this notebook file, I will be creating a more complex environment to address the limitations that the other environment had. 

____
### 1. Imports

In [1]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import itertools
import numpy as np
import random
from dataclasses import dataclass, field

____
### 2. Weaknesses of previous environment

#### 2.1 Linearity of queries
The original environment structured common SQL patterns as a simple array of blocks that are placed in a specified sequence.
```python
qn = ["SELECT", "COLUMN", "FROM", "WHERE", "WHERE_CONDITION", "END"]
```
- This approach does not support the implementation of subqueries as part of the common SQL patterns
To address this limitation, SQL patterns will now be represented as a tree data structure, implemented as a nested dictionary within python

```python
linear_qn = {"blocks": ["SELECT", "COLUMN", "FROM", "TABLE", "WHERE", "COLUMN", "EQUAL", "VALUE"], 
        "subqueries" : {}
        }
subquery_qn = {"blocks": ["SELECT", "COLUMN", "FROM", "TABLE", "WHERE", "COLUMN", "IN", "SUBQUERY_START", "SUBQUERY_END"], 
            "subqueries" : {7: {
                "blocks" : ["SELECT", "COLUMN", "FROM", "TABLE", "WHERE", "COLUMN", "OPERATOR", "VALUE"], 
                    "subqueries" : {}
                                }
                            }
            }
nested_subquery_qn = {'blocks': ['SELECT', 'COLUMN', 'FROM', 'TABLE', 'WHERE', 'COLUMN', 'IN', 'SUBQUERY_START', 'SUBQUERY_END'],
        'subqueries' : {7: {
            'blocks': ['SELECT', 'COLUMN', 'FROM', 'TABLE', 'WHERE', 'COLUMN', 'IN', 'SUBQUERY_START', 'SUBQUERY_END'],
                'subqueries' : {7: {
                    'blocks': ['SELECT', 'COLUMN', 'FROM', 'TABLE', 'WHERE', 'COLUMN', 'OPERATOR', 'VALUE'],
                        'subqueries': {}
                                    }
                                }
                            }
                        }
                    }
```

- Every dictionary contains 2 keys, `blocks` and `subqueries`
    - `blocks` is a list of blocks in proper sequence
    - `subqueries` is a nested dictionary in the same format with `blocks` and `subqueries` as keys if there is a subquery, else it is just an empty dictionary `{}`
    - the position of `SUBQUERT_START` and `SUBQUERY_END` in the `blocks` array indicates the position of where the `subquery` is at 
- `blocks` is mostly flat tokens, but wherever a `SUBQUERY_START` sits at some positon `i`, `blocks[i]` points to an entire other node of the same shape, a nested query
- from the example above:
    1. linear query: 0 nodes 
    2. one sub-query: 1 node
    3. two sub-query: 2 nodes

#### 2.2 Simple block design 
- The original environment had a simple design with limited blocks due to the simplicity of linear composition
- To address this limitation, the new environment redesigned the block systems, utilising vertical blocks for chaining and horizontal blocks to add details to each clause.

In [2]:
VERTICAL_BLOCKS = [
    "SELECT",
    "FROM",
    "WHERE",
    "GROUP_BY",
    "HAVING",
    "ORDER_BY",
    "LIMIT",
    "JOIN",
]

HORIZONTAL_BLOCKS = [
    "DISTINCT",
    "COLUMN",           # <column name>, optionally alias-qualified
    "STAR",              # (*)
    "TABLE",              # <table name>, optionally aliased
    "IS_NULL",
    "IS_NOT_NULL",
    "OPERATOR",           # dropdown: =, >=, <=, <, >, !=, <>
    "LIKE",
    "IN",
    "NOT",
    "AND",
    "OR",
    "VALUE",
    "AGG_FUNC",           # dropdown: COUNT, SUM, AVG, MIN, MAX -- wraps COLUMN/STAR
    "ASC",
    "DESC",
    "OFFSET",
    "ON",
    "SUBQUERY_START",     # opens a nested query -- pushes a new frame
    "SUBQUERY_END",       # closes it -- placed after the nested frame auto-pops
]

In [3]:
BLOCK_TYPES = VERTICAL_BLOCKS + HORIZONTAL_BLOCKS
BLOCK_TO_IDX = {b: i for i, b in enumerate(BLOCK_TYPES)}
IDX_TO_BLOCK = {i: b for b, i in BLOCK_TO_IDX.items()}
N_BLOCKS = len(BLOCK_TYPES)

MAX_SEQUENCE_LEN = 14   # padded length of the CURRENT (innermost) frame's built_seq
MAX_TOTAL_STEPS = 40    # episode-wide step budget across all frames combined
MAX_DEPTH = 3            # cap on subquery nesting depth (0 = top level)


#### 2.3 Simple block design 
- The original environment used `"END"` as a signal to show that the query is complete.
- However, that was a redundant signal as whether a frame was complete was already fully determined by `len(built_seq) == len(target)` 
- Removing `"END"` also changes the action space, where the model has to also determine if the block they have just placed is the last one

```python
self.action_space = spaces.MultiDiscrete([N_BLOCKS, 2])
```
- The action space is now a 2-dimensional array
    1. A number, representing the index of the block that the model chooses to place
    2. A binary indicator, indicating if the model thinks the block it just placed is the end of the query.

____
### 3. Gym Environment
- simulate a user constructing SQL queries using a block editor and providing suggestions for the next block given what has been placed so far

- Frame: a live, partially built copy of a tree node
    - no information of it parent or child node
    - the storage for the query being built, either top level ()
- State: a `stack` to store the individual frames
    - the item at the bottom of the stack, `stack[0]` is the top-level frame, ie the first node with no parent
    - the item at the top of the stack, `stack[-1]` is the frame that is still being built, ie the latest node that is currently still being filled in
    - everything else in between is a subquery
    - A new `Frame` is created and pushed into the stack when `SUBQUERY_START` is placed in the frame on the top of the stack 
    - When the top `Frame` is complete, it is then popped from the stack
    - Therefore, the `Frame` left below is 'frozen' in place with `SUBQUERY_START` as the latest block, until the subquery is complete and the next expected block is `SUBQUERY_END`
- action space : 1-dim
    - a number within range of `(0, 2 * N_BLOCKS)` which decodes to a `number indicating the block placed` and a `binary indicator` of whether the agent thinks the block that it just placed is the last block (Frame ends here)
- reward: environment rewards for 3 things, 
    1. correct syntax
        - -0.3 if SQL syntax is correct but wrong pattern, -2.0 if SQL syntax is wrong
    2. correct pattern
        - +3 if block is correct, else check syntaxing for exact reward
    3. correct end indicator
        - +2 if popped correctly, -2 if pops wrongly, when model thinks to stop the frame
        - +5 if ends the frame correctly, -3 if ends frame wrongly
        - -0.3 if SQL syntax is correct but wrong pattern, -2.0 if SQL syntax is wrong



#### 2.1 Defining SQL Grammar
- The SQL Grammar bank consists of a few components
1. `TRANSITIONS`
    - dictionaries nested inside a dictionary, where keys are referring to the current `stage` of the query and the inner dictionary contains the BLOCKS and the corresponding next stage of the query
    - `stage` is a label for "how far into a state am i", ie `"select_open"` means that `SELECT` was just placed so a select item is next
2. `COMPLETE_STATES`
    - a set containing all the possible completed stages where stopping at that point is legal SQL 

In [4]:
TRANSITIONS = {
    "start": {"SELECT": "select_open"},

    # --- SELECT item ---
    "select_open": {
        "DISTINCT": "select_after_distinct",
        "AGG_FUNC": "select_after_agg",
        "COLUMN": "select_item_done",
        "STAR": "select_item_done",
    },
    "select_after_distinct": {
        "AGG_FUNC": "select_after_agg",
        "COLUMN": "select_item_done",
        "STAR": "select_item_done",
    },
    "select_after_agg": {"COLUMN": "select_item_done", "STAR": "select_item_done"},
    "select_item_done": {"FROM": "after_from"},
    "after_from": {"TABLE": "main_table_done"},

    # --- After FROM TABLE: everything else is optional from here ---
    "main_table_done": {
        "JOIN": "after_join",
        "WHERE": "where_cond_open",
        "GROUP_BY": "after_groupby",
        "ORDER_BY": "after_orderby",
        "LIMIT": "after_limit",
    },

    # --- JOIN ---
    "after_join": {"TABLE": "join_table_done"},
    "join_table_done": {"ON": "after_on"},
    "after_on": {"COLUMN": "on_lhs_done"},
    "on_lhs_done": {"OPERATOR": "on_op_done"},
    "on_op_done": {"COLUMN": "join_done"},
    "join_done": {
        "WHERE": "where_cond_open",
        "GROUP_BY": "after_groupby",
        "ORDER_BY": "after_orderby",
        "LIMIT": "after_limit",
    },

    # --- WHERE condition automaton ---
    "where_cond_open": {"NOT": "where_after_not", "COLUMN": "where_cond_lhs_done"},
    "where_after_not": {"COLUMN": "where_cond_lhs_done"},
    "where_cond_lhs_done": {
        "OPERATOR": "where_cond_op_done",
        "IS_NULL": "where_cond_done",
        "IS_NOT_NULL": "where_cond_done",
        "LIKE": "where_cond_op_done",
        "IN": "where_cond_in",
    },
    "where_cond_op_done": {"VALUE": "where_cond_done"},
    "where_cond_in": {"SUBQUERY_START": "where_cond_subquery_open"},
    "where_cond_subquery_open": {"SUBQUERY_END": "where_cond_done"},
    "where_cond_done": {
        "AND": "where_cond_open",
        "OR": "where_cond_open",
        "GROUP_BY": "after_groupby",
        "ORDER_BY": "after_orderby",
        "LIMIT": "after_limit",
    },

    # --- GROUP BY / HAVING ---
    "after_groupby": {"COLUMN": "groupby_col_done"},
    "groupby_col_done": {
        "HAVING": "having_cond_open",
        "ORDER_BY": "after_orderby",
        "LIMIT": "after_limit",
    },
    "having_cond_open": {"AGG_FUNC": "having_after_agg", "COLUMN": "having_cond_lhs_done"},
    "having_after_agg": {"COLUMN": "having_cond_lhs_done"},
    "having_cond_lhs_done": {"OPERATOR": "having_cond_op_done"},
    "having_cond_op_done": {"VALUE": "having_cond_done"},
    "having_cond_done": {
        "AND": "having_cond_open",
        "OR": "having_cond_open",
        "ORDER_BY": "after_orderby",
        "LIMIT": "after_limit",
    },

    # --- ORDER BY ---
    "after_orderby": {"COLUMN": "orderby_col_done"},
    "orderby_col_done": {"ASC": "orderby_dir_done", "DESC": "orderby_dir_done", "LIMIT": "after_limit"},
    "orderby_dir_done": {"LIMIT": "after_limit"},

    # --- LIMIT / OFFSET ---
    "after_limit": {"OFFSET": "after_offset"},
    "after_offset": {},
}


COMPLETE_STATES = {
    "main_table_done", "join_done", "where_cond_done", "groupby_col_done",
    "having_cond_done", "orderby_col_done", "orderby_dir_done",
    "after_limit", "after_offset",
}

`TRANSITIONS[stage][token] = next_stage`
- if `token` isnt a key at all, that token is illegal at that point under any query and there is no valid next stage for it

In [5]:
def replay(built_seq):
    stage = "start"
    for tok in built_seq:
        stage = TRANSITIONS[stage][tok]
    return stage

- `replay(built_seq)` traces through the current sequence and finds the next stage 

In [6]:
def legal_next_blocks(built_seq):
    stage = replay(built_seq)
    return set(TRANSITIONS.get(stage, {}).keys())

- `legal_next_blocks(built_seq)` returns a set of the next possible blocks according to the current sequence

In [7]:
def valid_stop(built_seq):
    return replay(built_seq) in COMPLETE_STATES

- `valid_stop(built_seq)` returns a boolean to check if the current sequence can legally end at this stage

#### 2.2 Defining Problem Bank
1. to begin, arrays are initialised to store optional clause chunks, `None` means the clause is skipped 
    - `assemble()` concatenates the chunks together, forming the inital list of linear problems with no subqueries yet
    - `make_problem()` forms the problem in the form of a tree with nodes pointed at a subquery

In [8]:
SELECT_ITEMS = [ # things that can come after ITEMS
    ["COLUMN"],
    ["DISTINCT", "COLUMN"],
    ["STAR"],
    ["AGG_FUNC", "COLUMN"],
    ["AGG_FUNC", "STAR"],
]

JOINS = [ # things that can come after JOIN
    None,
    ["JOIN", "TABLE", "ON", "COLUMN", "OPERATOR", "COLUMN"],
]

# Where-variants WITHOUT a subquery (those are generated separately below,
# since they need a nested problem attached via `subqueries=`).
WHERE_VARIANTS = [
    None,
    ["WHERE", "COLUMN", "OPERATOR", "VALUE"],
    ["WHERE", "COLUMN", "IS_NULL"],
    ["WHERE", "COLUMN", "IS_NOT_NULL"],
    ["WHERE", "COLUMN", "LIKE", "VALUE"],
    ["WHERE", "NOT", "COLUMN", "LIKE", "VALUE"],
    ["WHERE", "COLUMN", "OPERATOR", "VALUE", "AND", "COLUMN", "OPERATOR", "VALUE"],
    ["WHERE", "COLUMN", "OPERATOR", "VALUE", "OR", "COLUMN", "OPERATOR", "VALUE"],
]

GROUPBY_VARIANTS = [
    None,
    ["GROUP_BY", "COLUMN"],
    ["GROUP_BY", "COLUMN", "HAVING", "AGG_FUNC", "COLUMN", "OPERATOR", "VALUE"],
]

ORDERBY_VARIANTS = [
    None,
    ["ORDER_BY", "COLUMN"],
    ["ORDER_BY", "COLUMN", "ASC"],
    ["ORDER_BY", "COLUMN", "DESC"],
]

LIMIT_VARIANTS = [
    None,
    ["LIMIT"],
    ["LIMIT", "OFFSET"],
]


In [9]:
def assemble(select_item, join, where, groupby, orderby, limit):
    blocks = ["SELECT"] + select_item + ["FROM", "TABLE"]
    for part in (join, where, groupby, orderby, limit):
        if part:
            blocks += part
    return blocks

In [10]:
def make_problem(blocks, subqueries=None):
    return {"blocks": blocks, "subqueries": subqueries or {}}

In [11]:
PROBLEM_BANK = []
for select_item, join, where, groupby, orderby, limit in itertools.product(
    SELECT_ITEMS, JOINS, WHERE_VARIANTS, GROUPBY_VARIANTS, ORDERBY_VARIANTS, LIMIT_VARIANTS
):
    blocks = assemble(select_item, join, where, groupby, orderby, limit)
    PROBLEM_BANK.append(make_problem(blocks))

2. `SIMPLE_SUBQUERY_POOL` is a smaller subquery-free set generated to fill the inside of subqueries, adding them to the `PROBLEM_BANK`

In [12]:
SIMPLE_SUBQUERY_POOL = []
for select_item, where in itertools.product(
    SELECT_ITEMS, WHERE_VARIANTS[:6]  # skip the AND/OR-chained variants, keep it simple
):
    blocks = assemble(select_item, None, where, None, None, None)
    SIMPLE_SUBQUERY_POOL.append(make_problem(blocks))

In [13]:
random.seed(0)  # deterministic bank generation
for select_item, join, groupby, orderby, limit in itertools.product(
    SELECT_ITEMS, JOINS, GROUPBY_VARIANTS, ORDERBY_VARIANTS[:2], LIMIT_VARIANTS[:2]
):
    where = ["WHERE", "COLUMN", "IN", "SUBQUERY_START", "SUBQUERY_END"]
    blocks = assemble(select_item, join, where, groupby, orderby, limit)
    subquery_pos = blocks.index("SUBQUERY_START")
    nested = random.choice(SIMPLE_SUBQUERY_POOL)
    PROBLEM_BANK.append(make_problem(blocks, subqueries={subquery_pos: nested}))

3. create roughly 20 problems with one more level of nesting subqueries and add them to `PROBLEM_BANK`

In [14]:
for _ in range(20):
    inner = random.choice(SIMPLE_SUBQUERY_POOL)
    mid_blocks = ["SELECT", "COLUMN", "FROM", "TABLE", "WHERE", "COLUMN", "IN", "SUBQUERY_START", "SUBQUERY_END"]
    mid = make_problem(mid_blocks, subqueries={7: inner})

    outer_blocks = ["SELECT", "COLUMN", "FROM", "TABLE", "WHERE", "COLUMN", "IN", "SUBQUERY_START", "SUBQUERY_END"]
    PROBLEM_BANK.append(make_problem(outer_blocks, subqueries={7: mid}))

conduct a simple validation check of `PROBLEM_BANK` against the grammar bank

In [15]:
def validate_problem(problem, path="root"):
    blocks = problem["blocks"]
    for i in range(1, len(blocks) + 1):
        prefix = blocks[:i]
        try:
            state = replay(prefix)
        except KeyError:
            raise AssertionError(f"[{path}] illegal prefix under grammar: {prefix}")
    final_state = replay(blocks)
    if final_state not in COMPLETE_STATES:
        raise AssertionError(f"[{path}] target ends in a non-terminal state: {blocks}")
    for idx, sub in problem.get("subqueries", {}).items():
        validate_problem(sub, path=f"{path}->sub@{idx}")

In [16]:
for idx, problem in enumerate(PROBLEM_BANK):
    validate_problem(problem, path=f"problem[{idx}]")

- checking the total number of problems generated

In [17]:
print(f"PROBLEM_BANK size: {len(PROBLEM_BANK)} problems")
print(f"SIMPLE_SUBQUERY_POOL size: {len(SIMPLE_SUBQUERY_POOL)}\n")

PROBLEM_BANK size: 3020 problems
SIMPLE_SUBQUERY_POOL size: 30



#### 2.3 Representation of Episode states
- `Frame` : a sequence in process, like a node within the full tree with no knowledge of its parents
- `EpisodeState` : a stack of `Frame`s where the latest item on the top is the active `Frame` being constructed and the `Frame` below is waiting for the `Frame` on top (subquery) to be complete
    - a new `Frame` is pushed into the stack stored inside `EpisodeState` when `SUBQUERY_START` is placed, representing a new query that is being built.

In [17]:
@dataclass
class Frame:
    target: dict # {"blocks": [...], "subqueries": {...}}
    built_seq: list = field(default_factory=list)

In [18]:
@dataclass
class EpisodeState:
    stack: list = field(default_factory=list)  # stack[-1] is the active (innermost) frame
    total_steps: int = 0  # steps across ALL frames, for truncation


#### 2.4 Representing the 2 actions in 1 number
- Since the DQN returns only 1 value, the 2 actions are encoded into 1 number and is decoded accordingly
- Number is within the range `(0, 2 * N_BLOCKS)`
- binary indicator : 1 or 0, indicating if the model thinks the current frame is complete
    - `indicator` = `action % 2` , ie indicator = 0 if action is even and indicator = 1 if action is odd
- block : index in range `(0, N_BLOCKS)` which corresponds to the block being placed, indicative from the dictionary `IDX_TO_BLOCK`
    - `index` = `action // 2`, ie removing the remainded when divided by 2 to 'remove' the binary indicator
    - `block` = `IDX_TO_BLOCK[index]` to convert the index to the string representation of the block 
- this decoding (through modulo and floor division)is handled by `divmod()`
    - `divmod(action, 2)` returns a tuple in the format `(action // 2, action % 2)` = `(index, indicator)`

#### 2.5 Creating Environment 

In [19]:
class SQLEnvAdvanced(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        obs_dim = MAX_SEQUENCE_LEN + 1 + N_BLOCKS + 1
        self.observation_space = spaces.Box(
            low=0, high=max(N_BLOCKS, MAX_SEQUENCE_LEN),
            shape=(obs_dim,), dtype=np.float32,
        )
        self.action_space = spaces.Discrete(N_BLOCKS * 2)
        self.state: EpisodeState | None = None

    @property
    def current_frame(self) -> Frame:
        return self.state.stack[-1]

    def required_blocks_vector(self):
        frame = self.current_frame
        target_list = frame.target["blocks"]
        remaining = target_list[len(frame.built_seq):]
        vec = np.zeros(N_BLOCKS, dtype=np.float32)
        for b in remaining:
            vec[BLOCK_TO_IDX[b]] = 1.0
        return vec

    def get_obs(self):
        frame = self.current_frame
        built_idx = [BLOCK_TO_IDX[b] + 1 for b in frame.built_seq]
        built_idx = built_idx[:MAX_SEQUENCE_LEN]
        built_idx += [0] * (MAX_SEQUENCE_LEN - len(built_idx))
        step_count = np.array([len(frame.built_seq) / MAX_SEQUENCE_LEN], dtype=np.float32)
        required = self.required_blocks_vector()
        depth = np.array([(len(self.state.stack) - 1) / MAX_DEPTH], dtype=np.float32)
        return np.concatenate([
            np.array(built_idx, dtype=np.float32),
            step_count,
            required,
            depth,
        ])

    def get_info(self):
        return {
            "depth": len(self.state.stack) - 1,
            "current_built_seq": list(self.current_frame.built_seq),
            "current_target_seq": list(self.current_frame.target["blocks"]),
            "total_steps": self.state.total_steps,
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        problem = random.choice(PROBLEM_BANK)
        self.state = EpisodeState(stack=[Frame(target=problem)], total_steps=0)
        return self.get_obs(), self.get_info()

    def step(self, action):
        block_idx, stop_flag = divmod(int(action), 2)
        block = self.idx_to_block(block_idx)
        self.state.total_steps += 1
        frame = self.current_frame
        target_list = frame.target["blocks"]
        pos = len(frame.built_seq)
        expected_block = target_list[pos] if pos < len(target_list) else None
        terminated = False
        truncated = False
        block_reward = 0.0
        stop_reward = 0.0
        block_correct = expected_block is not None and block == expected_block
        syntax_ok = block in legal_next_blocks(frame.built_seq)
        if block_correct:
            assert syntax_ok, f"target block {block!r} was flagged illegal -- grammar/bank bug" #debugging
            frame.built_seq.append(block)
            true_done = len(frame.built_seq) == len(target_list)
            if block == "SUBQUERY_START": #if supposed to start subquery, 
                stop_reward = 1.0 if stop_flag == 0 else -1.0
                if len(self.state.stack) - 1 < MAX_DEPTH:
                    nested_problem = frame.target["subqueries"][pos]
                    self.state.stack.append(Frame(target=nested_problem))
                    block_reward = 1.0
                else:
                    frame.built_seq.pop()
                    block_reward = -1.0
                    stop_reward = 0.0
            else:
                block_reward = 1.0
                stop_reward = 1.0 if stop_flag == int(true_done) else -1.0

                if stop_flag == 1:
                    if len(self.state.stack) > 1:
                        self.state.stack.pop()
                        block_reward += 2.0 if true_done else -2.0
                    else:
                        terminated = True
                        block_reward += 5.0 if true_done else -3.0
        else:
            block_reward = -0.3 if syntax_ok else -2.0
            stop_reward = -1.0 if stop_flag == 1 else 0.0
        reward = block_reward + stop_reward
        if self.state.total_steps >= MAX_TOTAL_STEPS:
            truncated = True
        return self.get_obs(), reward, terminated, truncated, self.get_info()

    def render(self):
        if self.render_mode == "human":
            for depth, frame in enumerate(self.state.stack):
                indent = "  " * depth
                print(f"{indent}depth {depth} built:  {frame.built_seq}")
                print(f"{indent}depth {depth} target: {frame.target['blocks']}")

    @staticmethod
    def idx_to_block(idx):
        return IDX_TO_BLOCK[idx]

##### 2.41 Initialisation 
```python
obs_dim = MAX_SEQUENCE_LEN + 1 + N_BLOCKS + 1
```
- first `MAX_SEQUENCE_LEN` numbers: built_seq of current frame (padded to MAX_SEQUENCE_LEN)
- next number : `step_count`within current frame (normalized to represent the percentage of completion)
- next `N_BLOCKS` numbers : the required remaining blocks in current frame (multi-hot over N_BLOCKS)
- last number : current nesting depth (normalized by `MAX_DEPTH`)

```python
self.action_space = spaces.Discrete(N_BLOCKS * 2)
```
- a single flat integer in the range of `[0, N_BLOCKS * 2)` which is decoded into 2 actions
- encoding: `flat = block_idx * 2 + stop_flag`
- decoded in `step()` via `block_idx, stop_flag = divmod(action, 2)`
- `block_idx` : index of the block type being placed (0 to N_BLOCKS-1)
- `stop_flag` : 0 or 1, indicating whether the agent believes this is the last block of the current frame

##### 2.42 helper to show current frame 
```python
@property
def current_frame(self) -> Frame:
    return self.state.stack[-1]
```
- returns the `Frame` at the top of the stack, which is representative of the query being built

##### 2.43 `required_blocks_vector()` - helper to construct the vector to represents the blocks required
```python
def required_blocks_vector(self):
    frame = self.current_frame
    target_list = frame.target["blocks"]
    remaining = target_list[len(frame.built_seq):]
    vec = np.zeros(N_BLOCKS, dtype=np.float32)
    for b in remaining:
        vec[BLOCK_TO_IDX[b]] = 1.0
    return vec
```
- same as the one in the original environment but checks for the `current_frame`


##### 2.43 `get_obs()` - to obtain the observation from the state

```python
def get_obs(self):
    frame = self.current_frame
    built_idx = [BLOCK_TO_IDX[b] + 1 for b in frame.built_seq]
    built_idx = built_idx[:MAX_SEQUENCE_LEN]
    built_idx += [0] * (MAX_SEQUENCE_LEN - len(built_idx))
    step_count = np.array([len(frame.built_seq) / MAX_SEQUENCE_LEN], dtype=np.float32)
    required = self.required_blocks_vector()
    depth = np.array([(len(self.state.stack) - 1) / MAX_DEPTH], dtype=np.float32)
    return np.concatenate([
        np.array(built_idx, dtype=np.float32),
        step_count,
        required,
        depth,
    ])
```
- same as the one in the previous environment but only views the `Frame` at the top of the stack, as seen by `frame = self.current_frame`
- the environment only shows the `Frame` that is currently being built to simulate a human only looking at the inner query while filling them in.



##### 2.44 `step()` - to progress the environment according to the action input 

```python
def step(self, action):
    block_idx, stop_flag = divmod(int(action), 2)
    block = self.idx_to_block(block_idx)
    self.state.total_steps += 1
    frame = self.current_frame
    target_list = frame.target["blocks"]
    pos = len(frame.built_seq)
    expected_block = target_list[pos] if pos < len(target_list) else None
    terminated = False
    truncated = False
    block_reward = 0.0
    stop_reward = 0.0
    block_correct = expected_block is not None and block == expected_block
    syntax_ok = block in legal_next_blocks(frame.built_seq)
```
- `block_idx, stop_flag = divmod(int(action), 2)` : decodes the flat Discrete action back into its two components
    - `block = self.idx_to_block(block_idx)` : converts index to the string version of the block 
- `frame = self.current_frame` : extracts the current frame as that is the one that is being checked
- `target_list = frame.target["blocks"]`, `pos = len(frame.built_seq)` : extracting information of the current frame
    - `target_list` : the completed version of the current `Frame`
    - `pos` : the progress of the current frame
- `expected_block = target_list[pos] if pos < len(target_list) else None` : the next block that is supposed to be placed
    - value of `None` if exceeds length of target
- `terminated = False`, `truncated = False`, `block_reward = 0.0`, `stop_reward = 0.0` : initialising default values
    - `block_reward` and `stop_reward` will add up to the total reward later on
- `block_correct = expected_block is not None and block == expected_block` : boolean to indicate if the block action is correct
- `syntax_ok = block in legal_next_blocks(frame.built_seq)` : boolean indicating if the syntax is correct

```python
if block_correct:
    assert syntax_ok, f"target block {block!r} was flagged illegal -- grammar/bank bug" 
    frame.built_seq.append(block)
    true_done = len(frame.built_seq) == len(target_list)
```
if the block action is correct, 
- `frame.built_seq.append(block)` : the updates the frame under `built_seq`
- `true_done = len(frame.built_seq) == len(target_list)` : boolean flag of whether the frame is at its correct length



```python
if block == "SUBQUERY_START": #if supposed to start subquery, 
    stop_reward = 1.0 if stop_flag == 0 else -1.0
    if len(self.state.stack) - 1 < MAX_DEPTH:
        nested_problem = frame.target["subqueries"][pos]
        self.state.stack.append(Frame(target=nested_problem))
        block_reward = 1.0
    else:
        frame.built_seq.pop()
        block_reward = -1.0
        stop_reward = 0.0
else:
    block_reward = 1.0
    stop_reward = 1.0 if stop_flag == int(true_done) else -1.0
    if stop_flag == 1:
        if len(self.state.stack) > 1:
            self.state.stack.pop()
            block_reward += 2.0 if true_done else -2.0
        else:
            terminated = True
            block_reward += 5.0 if true_done else -3.0
```
- `if block == "SUBQUERY_START"` : if the correct block to place is to start a subquery
- `stop_reward = 1.0 if stop_flag == 0 else -1.0` : correctly indicates that frame has not ended
    - reward = 1.0 if model chooses not to stop
    - reward = -1.0 if model places a `SUBQUERY_START` and decides the query ends here
- `if len(self.state.stack) - 1 < MAX_DEPTH` : the stack is not at `MAX_DEPTH` yet
    -  `nested_problem = frame.target["subqueries"][pos]` : finds the pattern of the subquery
    - `self.state.stack.append(Frame(target=nested_problem))` : pushing the new `Frame` onto the stack
    - `block_reward = 1.0` : reward for starting subquery properly
- `else, frame.built_seq.pop()` : the stack is at max length, not supposed to start subquery
    - `block_reward = -1.0` : wrong block placed
    - `stop_reward = 0.0` : no reward for stopping here
- `else` : correct block but not sub query
    - `block_reward = 1.0` : reward for correct block
    - `stop_reward = 1.0 if stop_flag == int(true_done) else -1.0` : reward for stopping properly is `1.0`, else `-1.0`
    - `if stop_flag == 1:` : if chooses to stop after placing this correct block, 
        - `if len(self.state.stack) > 1, self.state.stack.pop()` : pops the current `Frame`
        - `block_reward += 2.0 if true_done else -2.0` : reward for stopping correctly is extra +2.0, else -2.0 for early terminationg
        - `else terminated = True, block_reward += 5.0 if true_done else -3.0` : terminate correctly 


```python
else:
    block_reward = -0.3 if syntax_ok else -2.0
    stop_reward = -1.0 if stop_flag == 1 else 0.0
```
- `block_reward = -0.3 if syntax_ok else -2.0` : block reward according to syntax checks
    - wrong block, correct syntax, `-0.3`
    - wrong block, wrong syntax, `-2.0`
- `stop_reward = -1.0 if stop_flag == 1 else 0.0` : stop reward
    - wrong block, wrong stop, -1, else 0

```python
reward = block_reward + stop_reward
if self.state.total_steps >= MAX_TOTAL_STEPS:
    truncated = True
return self.get_obs(), reward, terminated, truncated, self.get_info()
```
- final steps before returning


#### 2.5 Exploring the environment 

In [20]:
env = SQLEnvAdvanced(render_mode="human")

def solve(problem):
    actions = []
    blocks = problem["blocks"]
    last_idx = len(blocks) - 1
    for i, b in enumerate(blocks):
        actions.append((b, 1 if i == last_idx else 0))
        if b == "SUBQUERY_START":
            actions.extend(solve(problem["subqueries"][i]))
    return actions

sample = random.sample(PROBLEM_BANK, 8)
total_solved = 0
for problem in sample:
    obs, info = env.reset()
    env.state.stack = [Frame(target=problem)]
    obs = env.get_obs()

    script = solve(problem)
    total_reward = 0.0
    terminated = False
    for block_name, stop_flag in script:
        # Encode (block_idx, stop_flag) → flat Discrete(N_BLOCKS*2) action
        action = BLOCK_TO_IDX[block_name] * 2 + stop_flag
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

status = "SOLVED" if terminated else "FAILED"
total_solved += int(terminated)
print(f"{status} | reward={total_reward:+.1f} | {problem['blocks']}")

print(f"\n{total_solved}/{len(sample)} sampled problems solved by scripted perfect agent.\n")

print("=== Wrong-but-legal vs wrong-and-illegal reward check ===")
problem = make_problem(["SELECT", "COLUMN", "FROM", "TABLE", "WHERE", "COLUMN", "OPERATOR", "VALUE"])
env.reset()
env.state.stack = [Frame(target=problem)]

    # Correct opening: SELECT, COLUMN, FROM, TABLE
for b in ["SELECT", "COLUMN", "FROM", "TABLE"]:
    env.step(BLOCK_TO_IDX[b] * 2 + 0)  # stop_flag=0

    # Now at main_table_done. Target wants WHERE next.
    # Try a WRONG-but-LEGAL move: GROUP_BY (legal here, just not the target).
obs, reward, term, trunc, info = env.step(BLOCK_TO_IDX["GROUP_BY"] * 2 + 0)
print(f"GROUP_BY here (wrong target, but legal SQL):  reward={reward:+.2f}")

    # Try a WRONG-and-ILLEGAL move: VALUE (not legal anywhere near here).
obs, reward, term, trunc, info = env.step(BLOCK_TO_IDX["VALUE"] * 2 + 0)
print(f"VALUE here (not legal SQL at all):            reward={reward:+.2f}")

SOLVED | reward=+47.0 | ['SELECT', 'AGG_FUNC', 'COLUMN', 'FROM', 'TABLE', 'JOIN', 'TABLE', 'ON', 'COLUMN', 'OPERATOR', 'COLUMN', 'WHERE', 'NOT', 'COLUMN', 'LIKE', 'VALUE', 'GROUP_BY', 'COLUMN', 'ORDER_BY', 'COLUMN', 'ASC']

1/8 sampled problems solved by scripted perfect agent.

=== Wrong-but-legal vs wrong-and-illegal reward check ===
GROUP_BY here (wrong target, but legal SQL):  reward=-0.30
VALUE here (not legal SQL at all):            reward=-2.00


### 3. Training the DQN model
- a Deep-Q-Network model is used in this case since the action space is discrete
- The classes are imported from my other repository : https://github.com/vruaaan/reinforcement-learning-projects

#### 3.1 Importing classes from outside 

In [21]:
from classes.dqn import DQNAgent

#### 3.2 Training and evaluating the agent

In [ ]:
env = SQLEnvAdvanced()
agent = DQNAgent(env=env, name="advanced-dqn")

In [25]:
agent.train(total_timesteps=500_000, learning_starts=200, log_every=10)

Timestep     289 | Episode   10 | Reward:  -102.00 | Epsilon: 0.995
Timestep     563 | Episode   20 | Reward:   -96.00 | Epsilon: 0.989
Timestep     888 | Episode   30 | Reward:   -30.00 | Epsilon: 0.983
Timestep    1120 | Episode   40 | Reward:    -9.00 | Epsilon: 0.979
Timestep    1397 | Episode   50 | Reward:   -14.00 | Epsilon: 0.973
Timestep    1744 | Episode   60 | Reward:   -87.00 | Epsilon: 0.967
Timestep    2130 | Episode   70 | Reward:   -90.30 | Epsilon: 0.960
Timestep    2384 | Episode   80 | Reward:   -92.60 | Epsilon: 0.955
Timestep    2712 | Episode   90 | Reward:   -89.90 | Epsilon: 0.948
Timestep    2963 | Episode  100 | Reward:   -96.00 | Epsilon: 0.944
Timestep    3211 | Episode  110 | Reward:   -17.00 | Epsilon: 0.939
Timestep    3535 | Episode  120 | Reward:   -96.00 | Epsilon: 0.933
Timestep    3822 | Episode  130 | Reward:   -99.00 | Epsilon: 0.927
Timestep    4177 | Episode  140 | Reward:   -94.60 | Epsilon: 0.921
Timestep    4419 | Episode  150 | Reward:   -77.

In [23]:
rewards = agent.evaluate(n_episodes=200)

Episodes: 200
Mean reward: -76.25 +/- 32.73
Reward range: [-120.00, -7.40]
Mean steps: 40.00


#### 3.3 Saving the model

In [ ]:
agent.save("organic.pth")

'organic.pth'